# GRPO Training Baseline

Train an offline GRPO-style RLVR adapter from the same 1.5B SFT g8 rollout data used by DAPO, then compare SFT vs DAPO vs GRPO on GSM8K test.

## 1. Pull Repo / Setup

In [ ]:
import importlib.util
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kdnehihi/strategy-distill-rl.git"
REPO_DIR = Path("/content/strategy-distill-rl")
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    if REPO_DIR.exists():
        print(f"Pulling latest repo in {REPO_DIR}")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)
    else:
        print(f"Cloning repo to {REPO_DIR}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    print(f"Local run detected. Current directory: {Path.cwd()}")

PROJECT_ROOT = Path.cwd()
print("Working directory:", PROJECT_ROOT)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


## 2. Install Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

AUTO_INSTALL_MISSING_DEPENDENCIES = True
REQUIRED_PACKAGES = {
    "peft": "peft",
    "accelerate": "accelerate",
    "transformers": "transformers",
    "tqdm": "tqdm",
    "pandas": "pandas",
}

missing = [pkg for import_name, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(import_name) is None]
if missing:
    if not AUTO_INSTALL_MISSING_DEPENDENCIES:
        raise ModuleNotFoundError("Missing packages: " + ", ".join(missing))
    print("Installing missing dependencies:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
else:
    print("All dependencies are available.")

# Older Colab torchao builds can break PEFT import. This project does not need torchao.
try:
    import torchao
    version = getattr(torchao, "__version__", "0.0.0")
    major_minor = tuple(int(part) for part in version.split(".")[:2] if part.isdigit())
    if major_minor and major_minor < (0, 16):
        print(f"Uninstalling incompatible torchao {version}")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
except Exception as exc:
    print("torchao check skipped:", exc)


## 3. Config

In [ ]:
from pathlib import Path

MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B-Instruct"
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/RL/Data")
DRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/RL/Checkpoints")

SFT_ZIP = DRIVE_CHECKPOINT_DIR / "balanced_r16_a32_4000_20260630_233638.zip"
DAPO_ZIP = DRIVE_CHECKPOINT_DIR / "dapo_1p5b_g8_lr5e7_e1_20260704_164638.zip"

SFT_ADAPTER_PATH = Path("checkpoints/student_sft/balanced_r16_a32_4000")
DAPO_ADAPTER_PATH = Path("checkpoints/student_dapo/dapo_1p5b_g8_lr5e7_e1")
GRPO_RUN_NAME = "grpo_1p5b_g8_lr5e7_e1"
GRPO_OUTPUT_DIR = Path("checkpoints/student_grpo") / GRPO_RUN_NAME

ROLLOUT_PATH = Path("data/rl_rollouts_1p5b_sft_g8.jsonl")
DRIVE_ROLLOUT_PATH = DRIVE_DATA_DIR / "rl_rollouts_1p5b_sft_g8.jsonl"
TEST_PATH = Path("data/gsm8k_clean_test.jsonl")
DRIVE_TEST_PATH = DRIVE_DATA_DIR / "gsm8k_clean_test.jsonl"

MAX_GROUPS = 100000
BATCH_SIZE = 1
EPOCHS = 1
LEARNING_RATE = 5e-7
CLIP_EPSILON = 0.2
MAX_LENGTH = 1024

RUN_GRPO_TRAINING = True
USE_REFERENCE_MODEL = True
SAVE_GRPO_TO_DRIVE = True

COMPARE_RUN_DIR = Path("runs/grpo_compare")
COMPARE_RUN_DIR.mkdir(parents=True, exist_ok=True)
COMPARE_EVAL_NUM_SAMPLES = -1
COMPARE_EVAL_BATCH_SIZE = 8
COMPARE_EVAL_MAX_NEW_TOKENS = 512

print("SFT zip:", SFT_ZIP)
print("DAPO zip:", DAPO_ZIP)
print("Rollout path:", ROLLOUT_PATH)
print("GRPO output:", GRPO_OUTPUT_DIR)


## 4. Helpers

In [ ]:
import json
import shutil
import subprocess
import zipfile
from collections import Counter
from datetime import datetime
from pathlib import Path

import pandas as pd


def mount_drive_if_needed():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive mount skipped or unavailable:", exc)


def run_command(args):
    command = [str(arg) for arg in args]
    print("$", " ".join(command))
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output_lines = []
    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)
    return_code = process.wait()
    if return_code != 0:
        tail = "".join(output_lines[-80:])
        raise RuntimeError(
            f"Command failed with exit code {return_code}: {' '.join(command)}\n"
            f"Last output lines:\n{tail}"
        )


def restore_zip_checkpoint(zip_path, target_dir, label):
    mount_drive_if_needed()
    zip_path = Path(zip_path)
    target_dir = Path(target_dir)
    if not zip_path.exists():
        raise FileNotFoundError(f"Missing {label} checkpoint zip: {zip_path}")
    if target_dir.exists():
        print(f"{label} checkpoint already restored: {target_dir}")
        return
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    print(f"Restoring {label} checkpoint from {zip_path} to {target_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(target_dir)


def copy_from_drive_if_missing(local_path, drive_path, label):
    mount_drive_if_needed()
    local_path = Path(local_path)
    drive_path = Path(drive_path)
    if local_path.exists():
        print(f"{label} already exists: {local_path}")
        return
    if not drive_path.exists():
        raise FileNotFoundError(f"Missing {label} in Drive: {drive_path}")
    local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, local_path)
    print(f"Copied {label}: {drive_path} -> {local_path}")


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def write_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def evaluate_adapter(run_name, adapter_path):
    output_path = COMPARE_RUN_DIR / f"{run_name}_outputs.jsonl"
    metrics_path = COMPARE_RUN_DIR / f"{run_name}_metrics.json"
    run_command([
        "python", "-B", "scripts/evaluate_student.py",
        "--model-name", MODEL_NAME,
        "--adapter-path", adapter_path,
        "--input-path", TEST_PATH,
        "--output-path", output_path,
        "--metrics-path", metrics_path,
        "--num-samples", COMPARE_EVAL_NUM_SAMPLES,
        "--batch-size", COMPARE_EVAL_BATCH_SIZE,
        "--max-new-tokens", COMPARE_EVAL_MAX_NEW_TOKENS,
    ])
    row = read_json(metrics_path)
    row["run"] = run_name
    row["metrics_path"] = str(metrics_path)
    row["output_path"] = str(output_path)
    return row


def load_outputs(run_name):
    return read_jsonl(COMPARE_RUN_DIR / f"{run_name}_outputs.jsonl")


def paired_summary(base_rows, candidate_rows, base_name, candidate_name):
    base = {row["id"]: row for row in base_rows}
    cand = {row["id"]: row for row in candidate_rows}
    ids = sorted(set(base) & set(cand))
    fixes = [i for i in ids if base[i].get("is_correct") != 1 and cand[i].get("is_correct") == 1]
    regressions = [i for i in ids if base[i].get("is_correct") == 1 and cand[i].get("is_correct") != 1]
    return {
        "base": base_name,
        "candidate": candidate_name,
        "common": len(ids),
        "fixes": len(fixes),
        "regressions": len(regressions),
        "net_gain": len(fixes) - len(regressions),
        "both_correct": sum(base[i].get("is_correct") == 1 and cand[i].get("is_correct") == 1 for i in ids),
        "both_wrong": sum(base[i].get("is_correct") != 1 and cand[i].get("is_correct") != 1 for i in ids),
    }


## 5. Restore Inputs

In [ ]:
restore_zip_checkpoint(SFT_ZIP, SFT_ADAPTER_PATH, "SFT")
restore_zip_checkpoint(DAPO_ZIP, DAPO_ADAPTER_PATH, "DAPO")
copy_from_drive_if_missing(ROLLOUT_PATH, DRIVE_ROLLOUT_PATH, "1.5B g8 rollout data")
copy_from_drive_if_missing(TEST_PATH, DRIVE_TEST_PATH, "GSM8K test data")


## 6. Train GRPO

In [ ]:
if RUN_GRPO_TRAINING:
    command = [
        "python", "-B", "scripts/train_grpo.py",
        "--model-name", MODEL_NAME,
        "--adapter-path", SFT_ADAPTER_PATH,
        "--rollout-path", ROLLOUT_PATH,
        "--output-dir", GRPO_OUTPUT_DIR,
        "--max-groups", MAX_GROUPS,
        "--batch-size", BATCH_SIZE,
        "--epochs", EPOCHS,
        "--learning-rate", LEARNING_RATE,
        "--clip-epsilon", CLIP_EPSILON,
        "--max-length", MAX_LENGTH,
    ]
    if USE_REFERENCE_MODEL:
        command.extend(["--reference-adapter-path", SFT_ADAPTER_PATH])
    else:
        command.append("--no-reference-model")
    run_command(command)
else:
    print("RUN_GRPO_TRAINING=False; skipping GRPO training.")


## 7. Inspect GRPO Metrics

In [ ]:
metrics_path = GRPO_OUTPUT_DIR / "grpo_metrics.json"
if metrics_path.exists():
    display(pd.DataFrame([read_json(metrics_path)]))
else:
    print("No GRPO metrics file yet:", metrics_path)


## 8. Evaluate SFT vs DAPO vs GRPO

In [ ]:
rows = []
rows.append(evaluate_adapter("sft_baseline", SFT_ADAPTER_PATH))
rows.append(evaluate_adapter("dapo", DAPO_ADAPTER_PATH))
rows.append(evaluate_adapter("grpo", GRPO_OUTPUT_DIR))

metrics_df = pd.DataFrame(rows)
metric_cols = [
    "run", "total", "accuracy", "loose_math_accuracy", "format_valid_rate", "usable_rate",
    "correct", "loose_correct", "format_valid", "usable", "metrics_path", "output_path",
]
display(metrics_df[metric_cols])

sft_rows = load_outputs("sft_baseline")
dapo_rows = load_outputs("dapo")
grpo_rows = load_outputs("grpo")
paired_df = pd.DataFrame([
    paired_summary(sft_rows, dapo_rows, "sft", "dapo"),
    paired_summary(sft_rows, grpo_rows, "sft", "grpo"),
    paired_summary(dapo_rows, grpo_rows, "dapo", "grpo"),
])
display(paired_df)

metrics_df.to_csv(COMPARE_RUN_DIR / "sft_dapo_grpo_metrics.csv", index=False)
paired_df.to_csv(COMPARE_RUN_DIR / "paired_summaries.csv", index=False)
print("Saved comparison files to", COMPARE_RUN_DIR)


## 9. Print GRPO Fixes and Regressions vs DAPO

In [ ]:
dapo = {row["id"]: row for row in load_outputs("dapo")}
grpo = {row["id"]: row for row in load_outputs("grpo")}
ids = sorted(set(dapo) & set(grpo))

grpo_fixes_vs_dapo = [i for i in ids if dapo[i].get("is_correct") != 1 and grpo[i].get("is_correct") == 1]
grpo_regressions_vs_dapo = [i for i in ids if dapo[i].get("is_correct") == 1 and grpo[i].get("is_correct") != 1]

print("GRPO fixes vs DAPO:", len(grpo_fixes_vs_dapo))
print("GRPO regressions vs DAPO:", len(grpo_regressions_vs_dapo))

for title, id_list in [("GRPO FIX VS DAPO", grpo_fixes_vs_dapo), ("GRPO REGRESSION VS DAPO", grpo_regressions_vs_dapo)]:
    print("#" * 120)
    print(title)
    print("#" * 120)
    for idx, id_ in enumerate(id_list[:30], start=1):
        print("=" * 100)
        print(f"sample #{idx} | id={id_}")
        print("question:", grpo[id_].get("question"))
        print("ground_truth:", grpo[id_].get("ground_truth"))
        print("\n--- DAPO ---")
        print("answer:", dapo[id_].get("model_answer"), "correct:", dapo[id_].get("is_correct"))
        print(dapo[id_].get("model_output"))
        print("\n--- GRPO ---")
        print("answer:", grpo[id_].get("model_answer"), "correct:", grpo[id_].get("is_correct"))
        print(grpo[id_].get("model_output"))


## 10. Save GRPO Checkpoint to Drive

In [ ]:
if SAVE_GRPO_TO_DRIVE:
    mount_drive_if_needed()
    if not GRPO_OUTPUT_DIR.exists():
        raise FileNotFoundError(f"GRPO output not found: {GRPO_OUTPUT_DIR}")
    DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    archive_base = Path("/tmp") / f"{GRPO_RUN_NAME}_{timestamp}"
    archive_path = shutil.make_archive(str(archive_base), "zip", GRPO_OUTPUT_DIR)
    drive_archive_path = DRIVE_CHECKPOINT_DIR / Path(archive_path).name
    shutil.copy2(archive_path, drive_archive_path)
    print("Saved GRPO checkpoint archive to:")
    print(drive_archive_path)
    print("Archive size MB:", drive_archive_path.stat().st_size / (1024 * 1024))
else:
    print("SAVE_GRPO_TO_DRIVE=False; skipping Drive archive.")
